# Zirkularitäts-Kontrolle: Partisan-Achse herausrechnen und neu clustern

Der harte Test gegen den Zirkelschluss. Aus jedem Subreddit-Vektor wird die Komponente entlang der
Partisan-Achse entfernt, dann wird auf diesen residualisierten Vektoren die komplette Clustering-
Pipeline neu gefahren. Danach werden die neuen Cluster mit dem URSPRUENGLICHEN Partisan-Score
bewertet.

**Testlogik.** Nach dem Herausrechnen ist die Achse im Raum null, eta2 auf ihr wäre trivial. Der
Test lautet deshalb: sortieren Cluster, die OHNE die politische Dimension gebaut wurden, trotzdem
politisch, und steigt das über die Zeit? Wenn ja, ist die Sortierung kein Bauartefakt der Achse.

**Ablauf.**
1. Achse bauen (identisch zu `Political_Axis_Analysis.ipynb`).
2. Alle neun Vektordateien residualisieren, in `Data/Vektoren_residual/` schreiben.
3. Angepasste Pipeline `Cluster_Vektoren_RESIDUAL.py` erzeugen (liest die residualisierten Vektoren).
4. **[HEAVY, WSL]** die Pipeline laufen lassen, Stunden, RAM-hungrig (AlignedUMAP).
5. Neue Cluster mit dem Original-Score bewerten, eta2-Trend gegen die Originalreihe.

**Ehrlicher Kontext.** Der Level-Placebo (Notebook Validierung) zeigte schon, dass die Partisan-Achse
ein unterdurchschnittlicher Cluster-Trenner ist, die Cluster also nicht um Politik gebaut sind. Dieser
Lauf dürfte das bestätigen. Der einzige wirklich neue Erkenntnisgewinn ist, ob der eta2-ANSTIEG die
Neuclusterung überlebt.

In [1]:
# ============================================================
# 1  Setup: Partisan-Achse bauen (identisch zu Political_Axis_Analysis)
# ============================================================
import os, numpy as np, pandas as pd
from gensim.models import KeyedVectors

def load_as_df(path):
    m = KeyedVectors.load_word2vec_format(path, binary=False, unicode_errors="ignore")
    v = pd.DataFrame(m.vectors, index=m.index_to_key)
    return v.divide(np.linalg.norm(v.values, axis=1), axis="rows")

def make_dim(seeds, vectors):
    kept = [(l, r) for l, r in seeds if l in vectors.index and r in vectors.index]
    R = vectors.loc[[r for l, r in kept]].values
    L = vectors.loc[[l for l, r in kept]].values
    return (R - L).sum(axis=0), len(kept)

clean_seeds = [
    ("Liberal", "Conservative"), ("progressive", "conservatives"),
    ("Democrat", "Republican"), ("Political_Revolution", "ConservativesOnly"),
    ("AskALiberal", "askaconservative"), ("AskDemocrats", "AskTrumpSupporters"),
    ("askhillarysupporters", "AskThe_Donald"), ("hillaryclinton", "The_Donald"),
    ("SandersForPresident", "HillaryForPrison"), ("Impeach_Trump", "HillaryMeltdown"),
]
df_16 = load_as_df("Data/Vektoren/vektoren_2016.txt")
axis_raw, n_pairs = make_dim(clean_seeds, df_16)
w = axis_raw / np.linalg.norm(axis_raw)          # Einheits-Achse im 2016-Raum
print("Achse gebaut aus %d Seed-Paaren, Dimension %d." % (n_pairs, w.shape[0]))

Achse gebaut aus 10 Seed-Paaren, Dimension 150.


## 2  Residualisieren

Für jede Eingabedatei der Pipeline (2016 plus die acht SeNSe-projizierten Jahre, alle im
2016-Raum) wird von jedem Vektor die Komponente entlang der Achse abgezogen, `v' = v - (v . w) w`.
Danach steht kein Vektor mehr in Achsenrichtung, die politische Dimension ist raus. Die Dateien
werden im gleichen word2vec-Format nach `Data/Vektoren_residual/` geschrieben, mit gleichen Namen,
damit die angepasste Pipeline sie findet.

In [3]:
# ============================================================
# 2  Residualisieren aller neun Vektordateien
# ============================================================
JAHRE = list(range(2016, 2025))
OUT_DIR = "Data/Vektoren_residual"
os.makedirs(OUT_DIR, exist_ok=True)

def src_path(j):
    if j == 2016:
        return "Data/Vektoren/vektoren_2016.txt"
    return f"SeNSe-main/SeNSe-main/output/projected_{str(j)[-2:]}_onto_16_FULL.txt"

def out_path(j):
    return (f"{OUT_DIR}/vektoren_2016.txt" if j == 2016
            else f"{OUT_DIR}/projected_{str(j)[-2:]}_onto_16_FULL.txt")

for j in JAHRE:
    kv = KeyedVectors.load_word2vec_format(src_path(j), binary=False, unicode_errors="ignore")
    V  = kv.vectors.astype(np.float64)
    Vr = V - np.outer(V @ w, w)                  # Komponente entlang w entfernen
    rest = float(np.abs(Vr @ w).max())           # sollte ~0 sein
    kv.vectors = Vr.astype(np.float32)
    kv.save_word2vec_format(out_path(j), binary=False)
    print(f"{j}: {V.shape[0]:6d} Vektoren residualisiert | max|v'.w| = {rest:.2e} -> {out_path(j)}")
print("\nFertig. Orthogonalitaet ok, wenn alle max|v'.w| ~ 0.")

2016:  16618 Vektoren residualisiert | max|v'.w| = 3.48e-07 -> Data/Vektoren_residual/vektoren_2016.txt
2017:  18426 Vektoren residualisiert | max|v'.w| = 2.00e-07 -> Data/Vektoren_residual/projected_17_onto_16_FULL.txt
2018:  20255 Vektoren residualisiert | max|v'.w| = 2.66e-07 -> Data/Vektoren_residual/projected_18_onto_16_FULL.txt
2019:  22409 Vektoren residualisiert | max|v'.w| = 2.64e-07 -> Data/Vektoren_residual/projected_19_onto_16_FULL.txt
2020:  24842 Vektoren residualisiert | max|v'.w| = 2.62e-07 -> Data/Vektoren_residual/projected_20_onto_16_FULL.txt
2021:  26930 Vektoren residualisiert | max|v'.w| = 2.45e-07 -> Data/Vektoren_residual/projected_21_onto_16_FULL.txt
2022:  28640 Vektoren residualisiert | max|v'.w| = 3.24e-07 -> Data/Vektoren_residual/projected_22_onto_16_FULL.txt
2023:  29446 Vektoren residualisiert | max|v'.w| = 3.13e-07 -> Data/Vektoren_residual/projected_23_onto_16_FULL.txt
2024:  28848 Vektoren residualisiert | max|v'.w| = 3.88e-07 -> Data/Vektoren_residua

## 3  Angepasste Pipeline erzeugen und laufen lassen  [HEAVY, WSL]

Die nächste Zelle schreibt `Cluster_Vektoren_RESIDUAL.py`, eine exakte Kopie von
`Cluster_Vektoren_LABEL2016.py` mit nur drei Änderungen: sie liest die residualisierten Vektoren,
schreibt nach `subreddit_clusters_RESIDUAL.csv` und vergleicht gegen die aktuelle LABEL2016-Partition
(ARI/AMI zeigen, wie stark das Herausrechnen die Clusterung verändert, hoch = kaum, also nicht
politisch gebaut).

Danach in der WSL-venv starten (Stunden, RAM-hungrig, nichts Grosses parallel):

```
wsl.exe -e bash -lc 'cd /mnt/c/Users/felix/Documents/Masterarbeit && ~/.virtualenvs/Masterarbeit/bin/python -u Cluster_Vektoren_RESIDUAL.py > log_residual.txt 2>&1'
```

In [4]:
# ============================================================
# 3  Cluster_Vektoren_RESIDUAL.py schreiben (Kopie mit residualisierten Eingaben)
# ============================================================
src = open("Cluster_Vektoren_LABEL2016.py", encoding="utf-8").read()
repl = [
    ('"Data/Vektoren/vektoren_2016.txt"',
     '"Data/Vektoren_residual/vektoren_2016.txt"'),
    ('f"SeNSe-main/SeNSe-main/output/projected_{str(jahr)[-2:]}_onto_16_FULL.txt"',
     'f"Data/Vektoren_residual/projected_{str(jahr)[-2:]}_onto_16_FULL.txt"'),
    ('OUT_CSV   = "subreddit_clusters_aligned_2016_2024_LABEL2016.csv"',
     'OUT_CSV   = "subreddit_clusters_RESIDUAL.csv"'),
    ('CMP_CSV   = "cluster_vergleich_LABEL2016.csv"',
     'CMP_CSV   = "cluster_vergleich_RESIDUAL.csv"'),
    ('ORIGINAL  = "subreddit_clusters_aligned_2016_2024.csv"          # nur LESEN',
     'ORIGINAL  = "subreddit_clusters_aligned_2016_2024_LABEL2016.csv"  # Vergleich vs. aktuell'),
]
for a, b in repl:
    assert a in src, "Textstelle nicht gefunden: " + a[:50]
    src = src.replace(a, b)
open("Cluster_Vektoren_RESIDUAL.py", "w", encoding="utf-8").write(src)
print("Geschrieben: Cluster_Vektoren_RESIDUAL.py")
print("Jetzt in der WSL-venv laufen lassen (siehe Markdown oben).")

Geschrieben: Cluster_Vektoren_RESIDUAL.py
Jetzt in der WSL-venv laufen lassen (siehe Markdown oben).


## 4  Neu-Scoring und eta2-Trend  [nach dem Pipeline-Lauf]

Sobald `subreddit_clusters_RESIDUAL.csv` existiert: die neue fixe 2016er Residual-Partition mit dem
URSPRUENGLICHEN Partisan-Score je Jahr bewerten und den eta2-Anstieg 2016 bis 2024 gegen die
Originalreihe halten (fix, Anstieg +0,062).

- Bleibt der Anstieg ähnlich, überlebt die politische Sortierung die Neuclusterung ohne die Achse.
  Das ist die harte Antwort auf den Zirkelschluss.
- Bricht er zusammen, wurde die Sortierung von der Achse in der Clusterung getragen.

In [5]:
# ============================================================
# 4  eta2-Trend der Residual-Cluster (Original-Score)
# ============================================================
RES = "subreddit_clusters_RESIDUAL.csv"
if not os.path.exists(RES):
    print("Noch keine", RES, "- erst die Pipeline (Zelle oben) laufen lassen.")
else:
    clr = pd.read_csv(RES)
    fix_res = clr[(clr.jahr == 2016) & (clr.cluster != -1)].set_index("subreddit")["cluster"]
    sc = (pd.read_csv("politischer_wandel_2016_2024.csv").rename(columns={pd.read_csv(
          "politischer_wandel_2016_2024.csv").columns[0]: "subreddit"})
          .set_index("subreddit"))

    def eta2(score, codes, k):
        g = score.mean(); sst = ((score - g) ** 2).sum()
        sums = np.bincount(codes, weights=score, minlength=k)
        cnts = np.bincount(codes, minlength=k)
        return (cnts * (sums / np.maximum(cnts, 1) - g) ** 2).sum() / sst

    print("Jahr |   n | eta2 (Residual-Cluster, Original-Score)")
    serie = {}
    for j in range(2016, 2025):
        col = f"score_{j}"
        d = pd.DataFrame({"cl": fix_res, "s": sc[col]}).dropna()
        codes = pd.factorize(d["cl"].values)[0]
        e = eta2(d["s"].values, codes, int(codes.max()) + 1)
        serie[j] = e
        print(f"{j} | {len(d):5d} | {e:.4f}")
    delta = serie[2024] - serie[2016]
    print(f"\nResidual-Anstieg 2016->2024 : {delta:+.4f}")
    print("Original (fixe Partition)    : +0.0621")
    print("Ähnlich = Sortierung überlebt das Herausrechnen der Achse.")

Jahr |   n | eta2 (Residual-Cluster, Original-Score)
2016 | 10733 | 0.1468
2017 | 10669 | 0.1754
2018 | 10649 | 0.1636
2019 | 10628 | 0.1904
2020 | 10594 | 0.1895
2021 | 10579 | 0.1952
2022 | 10558 | 0.1995
2023 | 10423 | 0.1875
2024 | 10286 | 0.2080

Residual-Anstieg 2016->2024 : +0.0613
Original (fixe Partition)    : +0.0621
Aehnlich = Sortierung ueberlebt das Herausrechnen der Achse.
